# Toy Models of Superposition

**Thesis.** A nonlinear model given *sparse* inputs will represent more features than it
has dimensions — *superposition* — and sparsity is the knob that turns it on. The cost is
*interference*.

This notebook rebuilds the core of Elhage et al. (2022), *Toy Models of Superposition*,
following the paper's own build: a linear ceiling, then a ReLU model on dense data, then
we crank sparsity and watch five features arrange themselves as a pentagon in two
dimensions. Everything here is synthetic — no language model, no Arabic — because the
goal is to understand the *mechanism*.

## Act 0 — The puzzle

In real networks, single neurons are often *polysemantic*: one neuron responds to several
unrelated things. Why would a model do that instead of giving each concept its own
neuron? The paper's answer is **superposition** — when features are rare (sparse), a model
can pack more of them than it has dimensions by storing them in overlapping directions,
paying a little *interference* in return. We'll reproduce that from scratch and find the
exact knob — sparsity — that decides whether it happens.

In [ ]:
# lib: imports
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

SEED = 0
torch.manual_seed(SEED)
print("torch", torch.__version__, "| seed", SEED)

## Act 1 — Features, and the linear ceiling

The paper models each feature as a direction in activation space. Our synthetic data
mirrors what it assumes real features look like: mostly-zero (sparse) and non-negative.
Each feature is zero with probability `S` (the sparsity) and otherwise uniform on
`[0, 1)`. Each feature also has an *importance* — a scalar weight on its share of the
loss — that decays across features, so the model has to make trade-offs.

In [ ]:
# lib: make_batch
def make_batch(n_features, sparsity, batch_size, generator=None):
    """Sparse synthetic features.

    Each entry is 0 with probability `sparsity`, otherwise uniform on [0, 1).
    Returns a tensor of shape [batch_size, n_features].
    """
    vals = torch.rand(batch_size, n_features, generator=generator)
    keep = torch.rand(batch_size, n_features, generator=generator) >= sparsity
    return vals * keep

In [ ]:
# lib: importance
def importance_weights(n_features, decay=0.7):
    """Importance Iᵢ = decay**i, shape [n_features]. Matches the paper's Iᵢ = 0.7^i."""
    return decay ** torch.arange(n_features, dtype=torch.float32)

In [ ]:
gen = torch.Generator().manual_seed(SEED)
demo = make_batch(n_features=5, sparsity=0.8, batch_size=8, generator=gen)
print("batch shape:", tuple(demo.shape), "| fraction nonzero:", (demo > 0).float().mean().item())
print("importance:", importance_weights(5).tolist())

### The model

We embed `n` features into `m < n` dimensions with a matrix `W` of shape `[m, n]` (column
`i` is feature `i`'s direction), then read them back out through `Wᵀ` and add a bias:

- **Linear model:** `x' = Wᵀ W x`. It can represent at most `m` features orthogonally —
  superposition is impossible by construction (`WᵀW` is invertible ⟺ no superposition).
- **ReLU model:** `x' = ReLU(Wᵀ W x + b)`. The non-linearity is what lets it tolerate the
  interference that superposition creates.

In [ ]:
# lib: toymodel
class ToyModel(torch.nn.Module):
    """Embed n features into m<n dims via W [m, n], read back through Wᵀ, add bias.

    forward: h = x @ W.T ; out = h @ W + b ; ReLU(out) if use_relu else out.
    """
    def __init__(self, n_features, n_hidden, use_relu=True):
        super().__init__()
        self.use_relu = use_relu
        self.W = torch.nn.Parameter(torch.empty(n_hidden, n_features))
        torch.nn.init.xavier_normal_(self.W)
        self.b = torch.nn.Parameter(torch.zeros(n_features))

    def forward(self, x):
        h = x @ self.W.T           # [B, m]
        out = h @ self.W + self.b  # [B, n]
        return F.relu(out) if self.use_relu else out

In [ ]:
m = ToyModel(n_features=5, n_hidden=2)
gen = torch.Generator().manual_seed(SEED)
xb = make_batch(5, sparsity=0.8, batch_size=4, generator=gen)
print("W:", tuple(m.W.shape), "| b:", tuple(m.b.shape), "| out:", tuple(m(xb).shape))

In [ ]:
# lib: train
def train(model, sparsity, importance, steps=10_000, lr=1e-3, batch_size=1024, seed=0):
    """Train `model` to reconstruct sparse features under importance-weighted MSE.

    Loss = mean over batch of Σᵢ importanceᵢ · (xᵢ − x'ᵢ)². Returns loss sampled every
    500 steps.
    """
    n_features = model.W.shape[1]
    gen = torch.Generator().manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    for step in range(steps):
        x = make_batch(n_features, sparsity, batch_size, generator=gen)
        out = model(x)
        loss = (importance * (x - out) ** 2).sum(dim=-1).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
        if step % 500 == 0:
            losses.append(loss.item())
    return losses

In [ ]:
# lib: feature_norms
def feature_norms(model):
    """L2 norm of each feature's embedding column (how strongly it is represented)."""
    return model.W.detach().norm(dim=0)

In [ ]:
imp5 = importance_weights(5)                       # Iᵢ = 0.7^i
lin = ToyModel(5, 2, use_relu=False)
train(lin, sparsity=0.0, importance=imp5, seed=SEED)
norms = feature_norms(lin)
print("linear / dense feature norms:", [round(v, 3) for v in norms.tolist()])
n_represented = int((norms > 0.5 * norms.max()).sum())
print("features clearly represented:", n_represented)
assert n_represented <= 2, "a 2-dim linear model should keep at most its top-2 features"

The linear model behaves like PCA: with two dimensions it keeps only the two most
important features and discards the rest. This is the ceiling — no non-linearity, so no
way to tolerate interference, so no superposition. Next we add the ReLU.

## Act 2 — Add the ReLU, keep the data dense

Does the ReLU alone create superposition? No. On dense data the ReLU model keeps the same
top-2 features the linear model did. The non-linearity only pays off once the data is
sparse — which is the whole point of Act 3.

In [ ]:
relu_dense = ToyModel(5, 2, use_relu=True)
train(relu_dense, sparsity=0.0, importance=imp5, seed=SEED)
norms = feature_norms(relu_dense)
print("ReLU / dense feature norms:", [round(v, 3) for v in norms.tolist()])
n_represented = int((norms > 0.5 * norms.max()).sum())
print("features clearly represented:", n_represented)
assert n_represented <= 2, "ReLU on DENSE data should still keep only its top-2 features"

Same story as the linear model: two features in, three discarded. The ReLU changed
nothing while the data is dense. Now we make the data sparse and watch it change
everything.